# Setup with LangChain, LamgSmith, and LangServe

- Get setup with Langchain, LangSmith, and LangServe
- Use the most and basic and common components of Langchain: prompt templates, models, and output parsers.
- Build a simple application with LangChain.
- Trace your application with LangSmith.
- Serve your application with LangServe

In [25]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [26]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY') ## for LangSmith Tracking
os.environ['LANGSMITH_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
  model = "gemini-2.5-flash"
)
print(llm)

model='models/gemini-2.5-flash' google_api_key=SecretStr('**********') client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000002A9A37DCFD0> default_metadata=() model_kwargs={}


In [28]:
resp = llm.invoke("Explain RAG (Retrieval-Augmented Generation) in one short paragraph.")
print(resp.content)

Retrieval-Augmented Generation (RAG) enhances Large Language Models (LLMs) by first retrieving relevant information from an external knowledge base or documents. This retrieved context is then provided alongside the user's query to the LLM, allowing it to generate a response that is more accurate, factual, and grounded in the latest or specific information, significantly reducing hallucinations and improving trustworthiness.


## ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
  [
    ("system", "You are an expert AI Engineer. Answer my questions."), ## in tuple format
    ("user", "{input}")
  ]
)

prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI Engineer. Answer my questions.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

## Other way of writing this:

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

prompt = ChatPromptTemplate.from_messages(
  [
    SystemMessagePromptTemplate.from_template("You are an expert AI Engineer. Answer my questions."),
    HumanMessagePromptTemplate.from_template("{input}")
  ]
)

prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI Engineer. Answer my questions.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [33]:
chain = prompt | llm   ## first go to prompt then go to LLM

In [34]:
response = chain.invoke({"input" : "Tell me about LangSmith."})   ## 'input' as the placeholder in prompt
print(response)

content='LangSmith is a **developer platform for building, debugging, evaluating, and monitoring Large Language Model (LLM) applications**. Developed by the creators of LangChain, it acts as a crucial MLOps tool specifically tailored for the unique challenges of LLM development.\n\nThink of it as the "Datadog" or "New Relic" for your LLM applications, but with integrated testing and evaluation capabilities that are essential for the non-deterministic nature of LLMs.\n\nHere\'s a breakdown of what LangSmith offers:\n\n---\n\n### **Why LangSmith? (The Problem it Solves)**\n\nDeveloping LLM applications is complex. Unlike traditional software:\n1.  **Non-deterministic Outputs:** LLMs don\'t always give the same answer for the same prompt.\n2.  **Black Box Nature:** It\'s hard to understand *why* an LLM responded a certain way or made a particular decision in a multi-step chain.\n3.  **Prompt Engineering:** Iterating on prompts and chain configurations is trial-and-error without proper too

In [35]:
type(response)

langchain_core.messages.ai.AIMessage

In [39]:
print(response.content)

LangSmith is a **developer platform for building, debugging, evaluating, and monitoring Large Language Model (LLM) applications**. Developed by the creators of LangChain, it acts as a crucial MLOps tool specifically tailored for the unique challenges of LLM development.

Think of it as the "Datadog" or "New Relic" for your LLM applications, but with integrated testing and evaluation capabilities that are essential for the non-deterministic nature of LLMs.

Here's a breakdown of what LangSmith offers:

---

### **Why LangSmith? (The Problem it Solves)**

Developing LLM applications is complex. Unlike traditional software:
1.  **Non-deterministic Outputs:** LLMs don't always give the same answer for the same prompt.
2.  **Black Box Nature:** It's hard to understand *why* an LLM responded a certain way or made a particular decision in a multi-step chain.
3.  **Prompt Engineering:** Iterating on prompts and chain configurations is trial-and-error without proper tools.
4.  **Evaluation Chal

## StrOutput Parser: for O/P parsing

In [ ]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

chain = prompt | llm | output_parser  ## first go to prompt then go to LLM then go to O/P parser

response = chain.invoke({"input" : "Tell me about LangSmith."})   ## 'input' as the placeholder in prompt
print(response)

LangSmith is a **developer platform for debugging, monitoring, and evaluating Large Language Model (LLM) applications**. It's built by the same team behind LangChain and is designed to be an indispensable tool for anyone building production-grade LLM-powered applications, especially those using complex chains and agents.

Think of it as the observability and testing layer for your AI applications, similar to how tools like DataDog or Sentry provide observability for traditional software, but specifically tailored for the unique challenges of LLMs.

---

### Why LangSmith Exists (The Problem It Solves)

Building reliable LLM applications is hard for several reasons:

1.  **Non-Determinism:** LLMs are not deterministic. The same prompt can yield different results.
2.  **Complexity:** Applications often involve multiple steps, tool calls, retrievals, and LLM interactions (chains and agents). It's hard to trace the flow.
3.  **Debugging Black Box:** When something goes wrong (or even right